# Contextual GP: SACGP / LCEAGP / LCEMGP

## 1. Contextual GPとは

Contextual GPは、複数のコンテキストに分かれた構造を利用するGPです。

このNotebookでは2つの工程コンテキストを考えます。

- `context_A`: 条件 `[a0, a1]`
- `context_B`: 条件 `[b0, b1]`

`SACGP` と `LCEAGP` は、コンテキスト別の報酬を直接観測せず、**合計された報酬**だけを観測するケースを扱います。

`LCEMGP` は逆に、各コンテキストの報酬を個別に観測できるmulti-output / multi-task型です。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import LCEAGP, LCEMGP, SACGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. SACGP / LCEAGP用の集約報酬データ

In [ ]:
def reward_a(X):
    return torch.sin(2 * torch.pi * X[:, 0]) + 0.4 * X[:, 1]

def reward_b(X):
    return 0.7 * torch.cos(2 * torch.pi * X[:, 0]) - 0.3 * X[:, 1]

n = 24
train_X = torch.rand(n, 4)

y_a = reward_a(train_X[:, [0, 1]])
y_b = reward_b(train_X[:, [2, 3]])
train_Y = (y_a + y_b + 0.03 * torch.randn(n)).unsqueeze(-1)

decomposition = {
    "context_A": [0, 1],
    "context_B": [2, 3],
}

print(train_X.shape, train_Y.shape)

## 3. SACGP

SACGPは各コンテキストを独立な加法成分として扱います。
コンテキスト間に明示的な相関を学習させない、比較的単純な構造です。

In [ ]:
sac = SACGP(
    train_X=train_X,
    train_Y=train_Y,
    train_Yvar=None,
    decomposition=decomposition,
)

print("supports_mll:", sac.supports_mll)
print("raw_train_X:", sac.raw_train_X.shape)
print("raw_train_Y:", sac.raw_train_Y.shape)

fit_gpytorch_mll(sac.make_mll())
sac.eval()

## 4. LCEAGP

LCEAGPは潜在context embeddingを学習し、コンテキスト間の類似性・相関を表現します。
`cat_feature_dict=None` の場合はcontext名に対応する内部カテゴリ表現が使われます。

In [ ]:
lcea = LCEAGP(
    train_X=train_X,
    train_Y=train_Y,
    train_Yvar=None,
    decomposition=decomposition,
    train_embedding=True,
)

print("supports_mll:", lcea.supports_mll)
print("raw_train_X:", lcea.raw_train_X.shape)
print("raw_train_Y:", lcea.raw_train_Y.shape)

fit_gpytorch_mll(lcea.make_mll())
lcea.eval()

## 5. SACGPとLCEAGPのposterior比較

In [ ]:
grid = torch.linspace(0.0, 1.0, 120)
test_X = torch.stack(
    [
        grid,
        torch.full_like(grid, 0.5),
        grid,
        torch.full_like(grid, 0.5),
    ],
    dim=-1,
)

with torch.no_grad():
    sac_post = sac.posterior(test_X)
    lcea_post = lcea.posterior(test_X)

sac_mean = sac_post.mean.squeeze(-1)
lcea_mean = lcea_post.mean.squeeze(-1)

truth = (
    reward_a(test_X[:, [0, 1]])
    + reward_b(test_X[:, [2, 3]])
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(grid, truth, label="真値")
plt.plot(grid, sac_mean, label="SACGP")
plt.plot(grid, lcea_mean, label="LCEAGP")
plt.xlabel("共通走査値")
plt.ylabel("集約報酬")
plt.legend()
plt.show()

## 6. LCEMGP: コンテキスト別報酬が観測できる場合

LCEMGPでは、各contextをtaskとしてlong-formatで表現します。

ここでは共通の決定変数 `[x0, x1]` に対して、

- task 0 = context_A
- task 1 = context_B

とします。

In [ ]:
base_X = torch.rand(20, 2)

X_a = torch.cat([base_X, torch.zeros(20, 1)], dim=-1)
X_b = torch.cat([base_X, torch.ones(20, 1)], dim=-1)

Y_a = (reward_a(base_X) + 0.03 * torch.randn(20)).unsqueeze(-1)
Y_b = (reward_b(base_X) + 0.03 * torch.randn(20)).unsqueeze(-1)

mt_X = torch.cat([X_a, X_b], dim=0)
mt_Y = torch.cat([Y_a, Y_b], dim=0)

context_cat_feature = torch.tensor([
    [0.0],
    [1.0],
])

lcem = LCEMGP(
    train_X=mt_X,
    train_Y=mt_Y,
    task_feature=2,
    context_cat_feature=context_cat_feature,
)

print("supports_mll:", lcem.supports_mll)
print("raw_train_X:", lcem.raw_train_X.shape)
print("raw_train_Y:", lcem.raw_train_Y.shape)

fit_gpytorch_mll(lcem.make_mll())
lcem.eval()

## 7. LCEMGPのcontext別posterior

In [ ]:
test_base = torch.stack(
    [grid, torch.full_like(grid, 0.5)],
    dim=-1,
)

with torch.no_grad():
    post = lcem.posterior(test_base, output_indices=[0, 1])
    mean = post.mean

mean_a = mean[..., 0]
mean_b = mean[..., 1]
truth_a = reward_a(test_base)
truth_b = reward_b(test_base)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(grid, truth_a, label="context_A 真値")
plt.plot(grid, mean_a, "--", label="context_A LCEMGP")
plt.plot(grid, truth_b, label="context_B 真値")
plt.plot(grid, mean_b, "--", label="context_B LCEMGP")
plt.xlabel("x0（x1=0.5で固定）")
plt.ylabel("context別報酬")
plt.legend()
plt.show()

## 8. 3モデルの使い分け

| モデル | 観測する出力 | context間の関係 |
|---|---|---|
| SACGP | 集約された1つの報酬 | 独立な加法構造 |
| LCEAGP | 集約された1つの報酬 | 潜在embeddingで相関を学習 |
| LCEMGP | context別の報酬 | multi-taskとしてcontext相関を学習 |

### 選択の目安

- context別の内訳が観測できない → `SACGP` または `LCEAGP`
- context同士の類似性を学習したい → `LCEAGP`
- context別の出力を直接観測できる → `LCEMGP`

Contextual GPは、単にカテゴリ変数があるだけの問題とは異なります。
「複数contextの構造を明示的に利用できるか」が採用判断のポイントです。